In [3]:
# GOAL :: Do familail relationship regression (FR-reg)

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM import frreg, tools
import importlib

In [2]:
source = "GS"

# Step 1. Load data

In [3]:
# parameters
info_fn = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/relative_information/relatives.formatted.info"

In [4]:
# relative information format
df_pair = pd.read_csv(info_fn, sep='\t')
df_pair.head()

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,18826,21244,50,36,F,F,0.750000
1,1,SB,different-sex-sibling,34422,23884,33,35,F,M,0.353553
2,1,PC,daughter-mother,79198,67531,66,44,F,F,0.500000
3,1,SB,daughter-sister,20399,67531,38,44,F,F,0.750000
4,1,SB,daughter-sister,67267,67531,43,44,F,F,0.750000


# Step 2. FR-reg

In [5]:
pheno_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/phenotype"
frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/frreg"

In [6]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

40

## Step 2.1 DOR level

In [20]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    try:
        df_frreg, msgs = frreg.familial_relationship_regression_DOR(df_mrg)
        
        if len(msgs) > 0:
            warning_dicts[pheno] = msgs
            continue
        
        df_frreg.to_csv(
            f"{frreg_path}/DOR/{pheno}.DOR.frreg",
            sep='\t',
            index=False
        )
    except:
        continue

100%|██████████| 40/40 [00:19<00:00,  2.01it/s]


## Step 2.2 REL level

In [26]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    
    df_frreg, msgs = frreg.familial_relationship_regression_REL(df_mrg)
    
    # if len(msgs) > 0:
    #     warning_dicts[pheno] = msgs
    #     continue
    
    # save results
    df_frreg.to_csv(
        f"{frreg_path}/REL/{pheno}.REL.frreg",
        sep='\t',
        index=False
    )

100%|██████████| 107/107 [02:21<00:00,  1.32s/it]
